# 🧬 PANACEE – Notebook 02 : Phase 2 – Fine-tuning Toxicité
## Classification multi-tâches sur Tox21

**Objectif :** Fine-tuner l'encodeur pré-entraîné pour prédire la toxicité
sur **12 cibles biologiques** simultanément (dataset Tox21).

**Mécanismes anti-surapprentissage :**
- 🔒 Encodeur gelé les N premières epochs (freeze_encoder_epochs)
- 🔓 Dégel progressif couche par couche (gradual unfreezing)
- 📊 Monitoring train/val AUC gap
- ⏹️ Early stopping sur ROC-AUC validation
- ⚖️ pos_weight per task (corrige le déséquilibre de classes)

**Référence :**
- Wu et al. (2018) *"MoleculeNet: A Benchmark for Molecular Machine Learning"*
- Xiong et al. (2020) *"Pushing the Boundaries of Molecular Representation with AttentiveFP"*

---
> **Durée estimée** : 1-2h sur GPU T4

In [ ]:
import sys, os
sys.path.insert(0, "/content/panacee")
os.chdir("/content/panacee")

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from config import DEVICE, PHASE2, CHECKPOINT_DIR, GNN_ARCHITECTURE

print(f"Device  : {DEVICE}")
print(f"Arch    : {GNN_ARCHITECTURE.upper()}")
print(f"\nConfig Phase 2 :")
for k, v in PHASE2.items():
    print(f"  {k:<30}: {v}")

## Étape 1 – Récupérer le checkpoint Phase 1

In [ ]:
PHASE1_CKPT = f"{CHECKPOINT_DIR}/phase1/panacee_phase1.pth"

if not os.path.exists(PHASE1_CKPT):
    print("⚠️  Checkpoint Phase 1 non trouvé en local.")
    print("   Option A : Lancer Notebook_01 d'abord")
    print("   Option B : Récupérer depuis Google Drive")
    
    from google.colab import drive
    drive.mount('/content/drive')
    drive_ckpt = "/content/drive/MyDrive/Panacee_Checkpoints/panacee_phase1.pth"
    
    if os.path.exists(drive_ckpt):
        import shutil
        os.makedirs(os.path.dirname(PHASE1_CKPT), exist_ok=True)
        shutil.copy(drive_ckpt, PHASE1_CKPT)
        print(f"✅ Checkpoint récupéré depuis Drive : {PHASE1_CKPT}")
    else:
        print("❌ Checkpoint introuvable sur Drive. Lancez d'abord le Notebook_01.")
else:
    ckpt = torch.load(PHASE1_CKPT, map_location="cpu", weights_only=False)
    print(f"✅ Checkpoint Phase 1 trouvé")
    print(f"   Epoch         : {ckpt.get('epoch', '?')}")
    print(f"   Meilleure loss: {ckpt.get('metrics', {}).get('val_loss', '?')}")

## Étape 2 – Téléchargement Tox21 (DeepChem)

In [ ]:
from data_loaders import download_dataset_deepchem
from config import DATA_DIR

print("Téléchargement de Tox21 via DeepChem...")
tox21_path = download_dataset_deepchem("tox21", save_dir=DATA_DIR)

if tox21_path and os.path.exists(tox21_path):
    df = pd.read_csv(tox21_path)
    print(f"\n✅ Tox21 téléchargé : {tox21_path}")
    print(f"   Shape      : {df.shape}")
    print(f"   Colonnes   : {list(df.columns[:5])} ...")
    print(f"\nAperçu :")
    display(df.head(3))
else:
    print("⚠️  Erreur de téléchargement. Utilisation de données de démonstration.")

In [ ]:
# Analyse du déséquilibre de classes
TOX21_TASKS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase", "NR-ER",
    "NR-ER-LBD", "NR-PPAR-gamma", "SR-ARE", "SR-ATAD5",
    "SR-HSE", "SR-MMP", "SR-p53",
]

# Déterminer les colonnes disponibles
available_tasks = [t for t in TOX21_TASKS if t in df.columns]
print(f"Tâches Tox21 disponibles : {len(available_tasks)}/12")

if available_tasks:
    pos_rates = {}
    for task in available_tasks:
        valid = df[task].dropna()
        pos   = (valid == 1).sum()
        total = len(valid)
        pos_rates[task] = pos / total * 100

    # Visualisation
    fig, ax = plt.subplots(figsize=(12, 5))
    tasks  = list(pos_rates.keys())
    values = list(pos_rates.values())
    colors = ["tomato" if v > 30 else "steelblue" for v in values]

    bars = ax.bar(tasks, values, color=colors, edgecolor="white", linewidth=1.5)
    ax.axhline(50, color="black", linestyle=":", alpha=0.5, label="50% (équilibré)")
    ax.set_title("Taux de positivité par tâche Tox21 (% molécules toxiques)", fontsize=13)
    ax.set_ylabel("% Positifs")
    ax.set_ylim(0, 60)
    ax.legend()

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=9)

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("/content/panacee/results/tox21_class_imbalance.png", dpi=150)
    plt.show()
    print("\n→ La couleur rouge indique un fort déséquilibre (compensé par pos_weight)")

## Étape 3 – Création des DataLoaders

In [ ]:
from data_loaders import make_tox_loaders
from config import PHASE2, DATA_DIR

train_loader, val_loader = make_tox_loaders(
    csv_path=tox21_path,
    batch_size=PHASE2["batch_size"],
    val_split=0.10,
    max_molecules=PHASE2.get("max_molecules", 8000),
)

print("DataLoaders Tox21 créés :")
print(f"  Entraînement : {len(train_loader.dataset):,} molécules")
print(f"  Validation   : {len(val_loader.dataset):,} molécules")
print(f"  Taille batch : {PHASE2['batch_size']}")

# Récupérer les pos_weights pour corriger le déséquilibre
pos_weight = train_loader.dataset.get_pos_weight().to(DEVICE)
print(f"\n  pos_weight (correction déséquilibre) :")
for i, (task, pw) in enumerate(zip(available_tasks, pos_weight.cpu().numpy())):
    print(f"    {task:12s}: {pw:.3f}")

## Étape 4 – Entraînement Phase 2

In [ ]:
from trainer import train_phase2

print("=" * 65)
print("LANCEMENT PHASE 2 – FINE-TUNING TOXICITÉ")
print("=" * 65)
print(f"Epochs      : {PHASE2['epochs']}")
print(f"LR encodeur : {PHASE2['lr_encoder']}")
print(f"LR tête     : {PHASE2['lr_head']}")
print(f"Gel encodeur : {PHASE2['freeze_encoder_epochs']} premiers epochs")
print()

phase2_ckpt = train_phase2(
    train_loader=train_loader,
    val_loader=val_loader,
    pretrained_encoder_path=PHASE1_CKPT,
    arch=GNN_ARCHITECTURE,
    num_tasks=len(available_tasks),
    pos_weight=pos_weight,
)

print(f"\n✅ Phase 2 terminée !")
print(f"   Checkpoint : {phase2_ckpt}")

## Étape 5 – Analyse des métriques de classification

In [ ]:
import json

history_path = f"{CHECKPOINT_DIR}/phase2/history_phase2.json"
with open(history_path) as f:
    hist = json.load(f)

epochs     = hist["epochs"]
train_loss = hist["train_losses"]
val_loss   = hist["val_losses"]
train_auc  = hist["train_aucs"]
val_auc    = hist["val_aucs"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs, train_loss, label="Train", lw=2, color="royalblue")
axes[0].plot(epochs, val_loss,   label="Val",   lw=2, linestyle="--", color="tomato")
axes[0].set_title("Loss BCE", fontsize=12)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC
axes[1].plot(epochs, train_auc, label="Train AUC", lw=2, color="royalblue")
axes[1].plot(epochs, val_auc,   label="Val AUC",   lw=2, linestyle="--", color="tomato")
axes[1].set_title("ROC-AUC", fontsize=12)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AUC")
axes[1].set_ylim(0.5, 1.0)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Gap AUC (diagnostic surapprentissage)
gap = [t - v for t, v in zip(train_auc, val_auc)]
axes[2].plot(epochs, gap, lw=2, color="darkorange")
axes[2].axhline(0.10, color="red", linestyle=":", label="Seuil modéré (0.10)")
axes[2].axhline(0.20, color="darkred", linestyle=":", label="Seuil sévère (0.20)")
axes[2].fill_between(epochs, gap, alpha=0.2, color="orange")
axes[2].set_title("Gap Train/Val AUC (surapprentissage)", fontsize=12)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Train AUC − Val AUC")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("Phase 2 – Analyse des métriques", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("/content/panacee/results/phase2_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nRésumé Phase 2 :")
print(f"  Epochs entraînés  : {len(epochs)}")
print(f"  Meilleur Val AUC  : {max(val_auc):.4f}")
print(f"  Epoch optimal     : {epochs[val_auc.index(max(val_auc))]}")
print(f"  Gap final T/V     : {gap[-1]:.4f}")

if max(val_auc) >= 0.80:
    print(f"\n  ✅ Excellent ! AUC > 0.80 sur Tox21")
elif max(val_auc) >= 0.70:
    print(f"\n  ⚠️  AUC correcte (0.70-0.80). Essayez plus d'epochs ou l'arch GPS.")
else:
    print(f"\n  ❌ AUC faible (<0.70). Vérifiez les données et l'encodeur Phase 1.")

## Étape 6 – Évaluation finale par tâche

In [ ]:
from gnn_models import build_encoder
from prediction_heads import ToxicityClassifier
from metrics import compute_classification_metrics, find_optimal_threshold
from config import HIDDEN_DIM, DROPOUT

# Charger le meilleur modèle
encoder = build_encoder(arch=GNN_ARCHITECTURE)
model   = ToxicityClassifier(encoder, num_tasks=len(available_tasks),
                              hidden_dim=HIDDEN_DIM, dropout=DROPOUT)
ckpt = torch.load(phase2_ckpt, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state"] if "model_state" in ckpt else ckpt, strict=False)
model = model.to(DEVICE).eval()

# Inférence sur validation
all_logits, all_targets = [], []
with torch.no_grad():
    for batch_data, labels in val_loader:
        logits = model(batch_data.to(DEVICE))
        all_logits.append(logits.cpu())
        all_targets.append(labels.cpu())

all_logits  = torch.cat(all_logits)
all_targets = torch.cat(all_targets)
metrics     = compute_classification_metrics(all_logits, all_targets)

print(f"=" * 50)
print(f"MÉTRIQUES GLOBALES (validation)")
print(f"=" * 50)
print(f"  ROC-AUC     : {metrics.roc_auc:.4f}")
print(f"  AUPRC       : {metrics.auprc:.4f}")
print(f"  F1          : {metrics.f1:.4f}")
print(f"  MCC         : {metrics.mcc:.4f}")
print(f"  Balanced Acc: {metrics.balanced_acc:.4f}")

In [ ]:
# Sauvegarde sur Google Drive
import shutil
drive_dir = "/content/drive/MyDrive/Panacee_Checkpoints"
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(phase2_ckpt, os.path.join(drive_dir, "panacee_phase2.pth"))
print(f"✅ Checkpoint Phase 2 sauvegardé sur Drive")
print("   → Passez maintenant au Notebook_03_Phase3_MultiProp.ipynb")